# Handwriting Recognition via ANN
This small notebook aims at solving the well-knows\n ANN demo: multiclass classification of hand-written digits, using the classic MNIST dataset.

## Dataset analisys and cleaning
The dataset was downloaded from kaggle.com. The training set consists of 60.000 28x28 greyscale images. The testing set is similar, but contains 10.000 images.
The original format is UBYTE-IDX3, and was parsed into an appropriate numpy array via a custom-made function available in ubyte_idx_reader.py.

In [1]:
#IMPORT DEPENDANCES
from DATA.ubyte_idx_reader import Mnistreader
import torch
from torch.utils.data import DataLoader
from torch import nn
import matplotlib.pyplot as plt
import random
import numpy as np
import torch.nn.functional as F

### Device check and selection

In [2]:
#check for CUDA availability for Torch and check the correct device is selected
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


### Random seed setting for reproducibility

In [3]:
#SEED SETTING
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

## Pytorch wrappers & initialization
The Mnistreader custom class reads the files well, but Pytorch only speaks Tensors, Datasets and Dataloaders. Let's implement Dataset correctly in order to link Pytorch to our dataset:

In [4]:

#custom dataset class, implements the methods required by pytorch
class Dataset:

    def __init__(self, set_path=None, labels_path=None): #initializes the dataset instance with the actual data from the Mnistreader library
        self.data = Mnistreader(set_path).data
        self.labels = Mnistreader(labels_path).data
        
    def __len__(self):          #returns the size of the data
        return  len(self.data)
        
    def __getitem__(self, idx): #returns a single element in a (point, label) tuple, given the index
        return(torch.tensor(self.data[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long))
        #NB: if ANY feature processing was to be done, it should be done here on query, NOT on the entire dataset, to take advantage of asyncronous operations.

## Hyperparameters
We define all the necessary parameters here, so it's easier to experiment:

In [ ]:
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-3
NUM_WORKERS = 0

We now need to create our Dataset combined with an instance of the Dataloader class to help us with batching, asyncronous loading of data into the VRAM and random shuffling:

In [ ]:
train_set_path = "DATA/train-images.idx3-ubyte"
train_labels_path = "DATA/train-labels.idx1-ubyte"

train_d = Dataset(train_set_path, train_labels_path)
train_dataloader = DataLoader(train_d, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, persistent_workers=True)

## Neural Network instancing
In Pytorch, NNs are defined by subclassing nn.Module.

Layers are defined inside the constructor (__init__).

The forward() method is needed to define the feed operations: layer by layer, with appropriate non-linear functions. We will use the "vanilla" definition, and later use nn.Sequential to compact things up.

In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self): #init contains the weights.
        super().__init__()
        self.flatten = nn.Flatten() #useful to have N-dim inputs flattened automatically during the forward pass

        #TODO convert in nn.Sequential to understand how it works
        self.l1 = nn.Linear(28*28,14*28)
        self.l2 = nn.Linear(14*28,14*14)
        self.l3 = nn.Linear(14*14, 7*14)
        self.l4 = nn.Linear(7*14, 7*7)
        self.l5 = nn.Linear(7*7,10)

    def forward(self, x):  #forward contains the operations to be performed on the instanced weights.
        #Operations need to be written IN ORDER.
        #forward needs to return logits, then other functions will use the appropriate loss.
        x = self.flatten(x) #flatten input
        x = self.l1(x);     #1st layer
        x = F.relu(x)       #act. function
        x = self.l2(x);     #...
        x = F.relu(x)
        x = self.l3(x);
        x = F.relu(x)
        x = self.l4(x);
        x = F.relu(x)
        x = self.l5(x);
        return x            #output

Let's instance the NN, load it into the GPU and see if the structure is the one we wanted:

In [8]:
model = NeuralNetwork().to(device) #instance the NN and load it into memory (preferably VRAM)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (l1): Linear(in_features=784, out_features=392, bias=True)
  (l2): Linear(in_features=392, out_features=196, bias=True)
  (l3): Linear(in_features=196, out_features=98, bias=True)
  (l4): Linear(in_features=98, out_features=49, bias=True)
  (l5): Linear(in_features=49, out_features=10, bias=True)
)


Let's now define the loss and the optimizer. We'll use SGD for now, and move later to ADAM.

In [9]:
loss_fn = nn.CrossEntropyLoss() #loss function for multi-class classification
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE) #optimizer, with the model's parameters and a (fixed for now)learning rate


## Training loop!
We now setup the training loop.'

LO LO GRA BA S!

logits -> loss -> grad reset -> backward -> step

In [ ]:
losses = []

for e in range(EPOCHS):
    for x,y in train_dataloader: #NB: the dataloader is an iterator, it will give us BATCH_SIZE number of elements each epoch.
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = loss_fn(logits,y)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print('EPOCH ' +str(e)+"/"+str(EPOCHS)+ ' LOSS: '+ str(losses[-1]))


In [ ]:
plt.plot(losses)
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.title('Loss curve')
plt.show()